# Repeated Stratified 5-Fold Cross-Validation (10 repeats x 5 folds, all 14 classifiers)

In [1]:
import ast
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
HORIZONS = [1,2,3,4,5]
SEEDS = list(range(10))  # each seed = one repeat, controlling that repeat's 5-fold assignment

pd.set_option('display.max_columns', None)

## 1. Load modeling dataset + feature list

In [2]:
df = pd.read_csv('modeling_dataset.csv')
COMORBID_FEATURE_COLS = [c for c in df.columns if c not in (
    ['person_id', 'sex', 'age', 'postcode', 'rurality', 'ses_irsd_decile', 'ses_missing',
     'incident_cvd', 'years_followup', 'split',
     'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
     'baseline_n_episodes', 'baseline_history_days']
    + [f'label_{h}y' for h in HORIZONS] + [f'eligible_{h}y' for h in HORIZONS]
)]
IS_GEO = 'rurality' in df.columns

UTILISATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']
if IS_GEO:
    NUMERIC_COLS = ['age'] + UTILISATION_COLS + ['ses_irsd_decile', 'ses_missing']
    CATEGORICAL_COLS = ['sex', 'rurality']
else:
    NUMERIC_COLS = ['age'] + UTILISATION_COLS
    CATEGORICAL_COLS = ['sex']

FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS + COMORBID_FEATURE_COLS
print(f'{len(FEATURE_COLS)} features ({len(COMORBID_FEATURE_COLS)} comorbidity-derived)')

def make_preprocessor(scaled):
    num_step = StandardScaler() if scaled else 'passthrough'
    return ColumnTransformer([
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_COLS),
        ('num', num_step, NUMERIC_COLS),
        ('bin', 'passthrough', COMORBID_FEATURE_COLS),
    ])

def load_best_params(file_prefix):
    res = pd.read_csv(f'{file_prefix}_results.csv')
    out = {}
    for _, row in res.iterrows():
        params = ast.literal_eval(row['best_params'])
        out[row['horizon']] = {k.replace('clf__', ''): v for k, v in params.items()}
    return out

19 features (12 comorbidity-derived)


## 2. Model configs

In [3]:
MODEL_CONFIGS = {
    'Logistic Regression': dict(file_prefix='logreg', cls=LogisticRegression,
                                 fixed_kwargs=dict(solver='saga', max_iter=5000), scaled=True, has_seed=True),
    'Random Forest':       dict(file_prefix='rf', cls=RandomForestClassifier,
                                 fixed_kwargs=dict(n_jobs=-1), scaled=False, has_seed=True),
    'XGBoost':             dict(file_prefix='xgboost', cls=XGBClassifier,
                                 fixed_kwargs=dict(eval_metric='logloss'), scaled=False, has_seed=True,
                                 needs_scale_pos_weight=True),
    'LightGBM':            dict(file_prefix='lightgbm', cls=LGBMClassifier,
                                 fixed_kwargs=dict(class_weight='balanced', verbose=-1), scaled=False, has_seed=True),
    'SVM':                 dict(file_prefix='svm', cls=SVC,
                                 fixed_kwargs=dict(probability=True), scaled=True, has_seed=True),
    'Decision Tree':       dict(file_prefix='dtree', cls=DecisionTreeClassifier,
                                 fixed_kwargs=dict(), scaled=False, has_seed=True),
    'Naive Bayes':         dict(file_prefix='nb', cls=GaussianNB,
                                 fixed_kwargs=dict(), scaled=True, has_seed=False),
    'KNN':                 dict(file_prefix='knn', cls=KNeighborsClassifier,
                                 fixed_kwargs=dict(), scaled=True, has_seed=False),
    'Bagging (DT base)':   dict(file_prefix='bag_dt', cls=BaggingClassifier,
                                 base_kwargs=dict(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE)),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=False, has_seed=True),
    'Extra Trees':         dict(file_prefix='extratrees', cls=ExtraTreesClassifier,
                                 fixed_kwargs=dict(n_jobs=-1), scaled=False, has_seed=True),
    'Bagging (KNN base)':  dict(file_prefix='bag_knn', cls=BaggingClassifier,
                                 base_kwargs=dict(estimator=KNeighborsClassifier()),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=True, has_seed=True),
    'AdaBoost':            dict(file_prefix='adaboost', cls=AdaBoostClassifier,
                                 fixed_kwargs=dict(), scaled=False, has_seed=True),
    'Stacking (diverse)':  dict(file_prefix='stack_diverse', cls=StackingClassifier,
                                 base_kwargs=dict(
                                     estimators=[
                                         ('lr', LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE)),
                                         ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
                                         ('svm', SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE)),
                                     ],
                                     final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=True, has_seed=False),
    'Stacking (boosting)': dict(file_prefix='stack_boosting', cls=StackingClassifier,
                                 base_kwargs=dict(
                                     estimators=[
                                         ('xgb', XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')),
                                         ('lgbm', LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)),
                                         ('ada', AdaBoostClassifier(random_state=RANDOM_STATE)),
                                     ],
                                     final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=True, has_seed=False),
}

## 3. Run all 14 models x horizons x 10 repeats x 5 folds

In [4]:
all_summaries = []
all_raw = []

for model_name, cfg in MODEL_CONFIGS.items():
    best_params_by_horizon = load_best_params(cfg['file_prefix'])
    preprocessor = make_preprocessor(cfg['scaled'])

    for h in HORIZONS:
        elig_col, label_col = f'eligible_{h}y', f'label_{h}y'
        sub = df[df[elig_col]].copy()
        X_all, y_all = sub[FEATURE_COLS], sub[label_col].astype(int)

        base_params = dict(best_params_by_horizon[f'{h}y'])
        base_params.update(cfg['fixed_kwargs'])

        fold_metrics = []
        for seed in SEEDS:
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
                X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
                y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]

                run_params = dict(base_params)
                if cfg.get('needs_scale_pos_weight'):
                    run_params['scale_pos_weight'] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                if cfg['has_seed']:
                    run_params['random_state'] = seed

                clf_obj = cfg['cls'](**cfg.get('base_kwargs', {}))
                pipe = Pipeline([('preprocess', preprocessor), ('clf', clf_obj)])
                pipe.set_params(**{f'clf__{k}': v for k, v in run_params.items()})
                pipe.fit(X_train, y_train)
                proba = pipe.predict_proba(X_test)[:, 1]
                pred = (proba >= 0.5).astype(int)

                row = dict(model=model_name, horizon=f'{h}y', seed=seed, fold=fold_idx,
                           auc=roc_auc_score(y_test, proba) if y_test.nunique() > 1 else np.nan,
                           accuracy=accuracy_score(y_test, pred),
                           precision=precision_score(y_test, pred, zero_division=0),
                           recall=recall_score(y_test, pred, zero_division=0),
                           f1=f1_score(y_test, pred, zero_division=0))
                fold_metrics.append(row)
                all_raw.append(row)

        mdf = pd.DataFrame(fold_metrics)
        summary = dict(model=model_name, horizon=f'{h}y', n_evaluations=len(fold_metrics))
        for metric in ['auc', 'accuracy', 'precision', 'recall', 'f1']:
            summary[f'{metric}_mean'] = round(mdf[metric].mean(), 3)
            summary[f'{metric}_std'] = round(mdf[metric].std(), 3)
        all_summaries.append(summary)
        print(f"{model_name:22s} {h}y: AUC {summary['auc_mean']:.3f} +/- {summary['auc_std']:.3f}  (n={len(fold_metrics)})")

Logistic Regression    1y: AUC 0.723 +/- 0.040  (n=50)


Logistic Regression    2y: AUC 0.714 +/- 0.034  (n=50)


Logistic Regression    3y: AUC 0.704 +/- 0.030  (n=50)


Logistic Regression    4y: AUC 0.694 +/- 0.024  (n=50)


Logistic Regression    5y: AUC 0.653 +/- 0.032  (n=50)


...

Stacking (boosting)    1y: AUC 0.736 +/- 0.032  (n=50)


Stacking (boosting)    2y: AUC 0.709 +/- 0.032  (n=50)


Stacking (boosting)    3y: AUC 0.706 +/- 0.030  (n=50)


Stacking (boosting)    4y: AUC 0.700 +/- 0.024  (n=50)


Stacking (boosting)    5y: AUC 0.665 +/- 0.033  (n=50)


## 4. Save results

In [5]:
raw_df = pd.DataFrame(all_raw)
raw_df.to_csv('repeated_cv_raw_per_fold.csv', index=False)

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv('repeated_cv_summary.csv', index=False)
summary_df

,model,horizon,n_evaluations,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,Logistic Regression,1y,50,0.723,0.040,0.696,0.018,0.182,0.018,0.609,0.067,0.280,0.028
1,Logistic Regression,2y,50,0.714,0.034,0.678,0.019,0.242,0.020,0.585,0.058,0.342,0.028
2,Logistic Regression,3y,50,0.704,0.030,0.663,0.023,0.325,0.025,0.592,0.049,0.420,0.030
3,Logistic Regression,4y,50,0.694,0.024,0.651,0.020,0.413,0.023,0.570,0.047,0.479,0.027
4,Logistic Regression,5y,50,0.653,0.032,0.607,0.029,0.565,0.036,0.545,0.043,0.554,0.030
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Stacking (boosting),1y,50,0.736,0.032,0.903,0.002,0.194,0.375,0.007,0.013,0.012,0.025
66,Stacking (boosting),2y,50,0.709,0.032,0.858,0.003,0.435,0.437,0.016,0.017,0.030,0.033
67,Stacking (boosting),3y,50,0.706,0.030,0.797,0.006,0.643,0.214,0.048,0.025,0.087,0.043
68,Stacking (boosting),4y,50,0.700,0.024,0.732,0.013,0.582,0.081,0.187,0.035,0.281,0.043
